# MIXX - Food Demand Prediction and Waste Analytics

This notebook is the analysis companion to the Streamlit dashboard. It is organized to show the full project workflow in a clean way: data wrangling, exploratory analysis, feature engineering, model benchmarking, final model selection, and forecasting demos.

## Notebook Scope

This notebook keeps only the analysis that supports the overall project.

- It uses the same Python modules that power the dashboard.
- It removes duplicated and unrelated code.
- It keeps comments so each step is easy to understand.
- It avoids mixing Streamlit UI code with the machine learning analysis.

In [ ]:
from __future__ import annotations

# Standard library imports used to resolve paths cleanly.
from pathlib import Path
import sys

# Core analysis libraries.
import pandas as pd
from IPython.display import display

# Make sure the project root is importable when the notebook is opened from the repo root.
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Import the same project modules used by the Streamlit app.
from mixx import (
    DEFAULT_SOURCE_DATA_PATH,
    add_daily_data_and_predict,
    benchmark_regression_models,
    build_daily_input_frame,
    build_holiday_calendar,
    build_waste_summary,
    load_dataset,
    predict_next_day_all_dishes_smart,
    predict_next_day_all_dishes_with_forecast,
)
from mixx.constants import DEFAULT_SPLIT_DATE
from mixx.modeling import prepare_model_frame
from mixx.project_analysis import (
    build_data_profile,
    build_experiment_trail,
    build_exploratory_findings,
    choose_final_model,
    describe_analysis_process,
    describe_feature_choices,
    render_analysis_report,
)

# Prefer the runtime CSV when it exists, otherwise fall back to the seed dataset.
DATASET_PATH = Path('data/runtime/restaurant_data.csv')
if not DATASET_PATH.exists():
    DATASET_PATH = Path(DEFAULT_SOURCE_DATA_PATH)

print(f'Using dataset: {DATASET_PATH}')

## 1. Load and Validate the Dataset

The first step is to load the restaurant dataset using the same cleaning logic as the production app. This keeps the analysis and the dashboard consistent.

In [ ]:
# Load the cleaned dataset.
df = load_dataset(DATASET_PATH)

# Print a few high-level checks so the dataset structure is obvious.
print(f'Rows: {len(df)}')
print(f'Dishes: {df["dish_name"].nunique()}')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Columns: {list(df.columns)}')

# Show the first rows so the raw structure is easy to inspect.
display(df.head())

## 2. Exploratory Analysis

Before modeling, the project checks the size of the dataset, waste behavior, dish-level demand behavior, and external drivers such as temperature and holidays.

In [ ]:
# Build a compact profile of the dataset for quick understanding.
profile = build_data_profile(df)
profile_df = pd.DataFrame(
    {'Metric': list(profile.keys()), 'Value': list(profile.values())}
)

display(profile_df)

In [ ]:
# Build a daily waste summary and dish-level average demand summary.
waste_summary = build_waste_summary(df)

dish_summary = (
    df.groupby('dish_name', as_index=False)
    .agg(
        avg_sold_qty=('sold_qty', 'mean'),
        avg_cooked_qty=('cooked_qty', 'mean'),
        avg_wasted_qty=('wasted_qty', 'mean'),
    )
    .sort_values('avg_sold_qty', ascending=False)
)

print('Recent waste summary:')
display(waste_summary.tail(10))

print('Dish-level averages:')
display(dish_summary.round(2))

In [ ]:
# Convert the exploratory findings into a readable table.
findings_df = pd.DataFrame(
    {'Exploratory finding': build_exploratory_findings(df)}
)

display(findings_df)

## 3. Step-by-Step Analysis Process

This section makes the workflow explicit so the reasoning is easy to follow.

In [ ]:
# Show the end-to-end project workflow as a structured table.
analysis_steps_df = pd.DataFrame(
    [
        {'Step': step.title, 'Why it matters': step.detail}
        for step in describe_analysis_process()
    ]
)

display(analysis_steps_df)

## 4. Feature Engineering

The forecasting model uses time-series and external features that capture recent demand, weekly seasonality, rolling trend, weather, and holiday effects.

In [ ]:
# Build the prepared modeling frames used for regression.
prepared_df, model_df = prepare_model_frame(df)

# Show how many rows remain after feature engineering and null removal.
print(f'Prepared rows before dropping null feature rows: {len(prepared_df)}')
print(f'Model-ready rows after feature engineering: {len(model_df)}')

# Show a sample of the engineered features used for training.
feature_preview = model_df[
    [
        'date',
        'dish_name',
        'sold_qty',
        'target_next_day',
        'lag_1',
        'lag_7',
        'rolling_mean_7',
        'temperature_c',
        'is_holiday',
    ]
].head(10)

display(feature_preview)

In [ ]:
# Use the same time-based train/test split as the production benchmarking code.
split_ts = pd.to_datetime(DEFAULT_SPLIT_DATE)
train_rows = model_df[model_df['date'] < split_ts]
test_rows = model_df[model_df['date'] >= split_ts]

print(f'Train rows: {len(train_rows)}')
print(f'Test rows: {len(test_rows)}')
print(f'Split date: {split_ts.date()}')

In [ ]:
# Explain why each feature was chosen for the final workflow.
feature_choices_df = pd.DataFrame(
    [
        {'Feature': choice.name, 'Reason for inclusion': choice.reason}
        for choice in describe_feature_choices()
    ]
)

display(feature_choices_df)

## 5. Model Benchmarking

The project compares a naive baseline, Linear Regression, and Random Forest using the same train/test split. This keeps the comparison fair and reproducible.

In [ ]:
# Benchmark the candidate models.
benchmark_df = benchmark_regression_models(df)

display(benchmark_df)

In [ ]:
# Add a narrative experiment trail so the benchmark results are not just numbers.
experiment_df = pd.DataFrame(
    [
        {
            'Model': note.model,
            'MAE': round(note.metrics['MAE'], 2),
            'RMSE': round(note.metrics['RMSE'], 2),
            'MAPE (%)': round(note.metrics['MAPE (%)'], 2),
            'Takeaway': note.takeaway,
        }
        for note in build_experiment_trail(df)
    ]
)

display(experiment_df)

## 6. Final Model Selection

The final model is chosen not only by benchmark performance, but also by how easy it is to explain and deploy.

In [ ]:
# Summarize the final model choice in a recruiter-friendly format.
final_model_df = pd.DataFrame([choose_final_model(df)])

display(final_model_df)

## 7. Forecasting Demo

This section shows the two forecast views used by the project: a baseline forecast and a weather-aware forecast.

In [ ]:
# Generate the baseline next-day forecast from historical demand patterns only.
baseline_forecast = predict_next_day_all_dishes_smart(df)

print('Baseline next-day forecast:')
display(baseline_forecast)

In [ ]:
# Generate a weather-aware forecast using the last known temperature as a simple demo input.
next_date = pd.to_datetime(df['date'].max()) + pd.Timedelta(days=1)
last_known_temp = float(df.sort_values('date')['temperature_c'].iloc[-1])

weather_forecast = predict_next_day_all_dishes_with_forecast(
    df,
    temp_tomorrow_c=last_known_temp,
    date_tomorrow=next_date,
    holiday_calendar=build_holiday_calendar(df),
)

print('Weather-aware next-day forecast:')
display(weather_forecast)

## 8. Daily Update Simulation

The Streamlit dashboard lets a user enter daily actuals. This notebook demonstrates the same underlying update logic with a simple simulated example.

In [ ]:
# Start from the latest dish rows so the simulation uses realistic dish names and metadata.
entry_template = build_daily_input_frame(df)

# Create a simple example where the kitchen cooks slightly more than the sold amount.
sold_dict = {
    row['dish_name']: int(row['sold_qty'])
    for _, row in entry_template.iterrows()
}
cooked_dict = {
    dish_name: sold_qty + 5
    for dish_name, sold_qty in sold_dict.items()
}

# Simulate adding one more day of actuals and refreshing the next forecast.
simulated_date = pd.to_datetime(df['date'].max()) + pd.Timedelta(days=1)
simulated_temp = float(df.sort_values('date')['temperature_c'].iloc[-1])

updated_df, updated_dashboard, tomorrow = add_daily_data_and_predict(
    df=df,
    date_today=simulated_date,
    temp_today_c=simulated_temp,
    sold_dict=sold_dict,
    cooked_dict=cooked_dict,
)

print(f'Simulated update date: {simulated_date.date()}')
print(f'New forecast date: {tomorrow}')
display(updated_dashboard)

## 9. Full Narrative Report

The project also provides a generated narrative report in Python so the analysis can be reused outside the notebook.

In [ ]:
# Render the full narrative report generated from the analysis module.
report_text = render_analysis_report(df)
print(report_text)